# 🧪 Exploration des données (Couche Bronze)

Ce notebook permet de visualiser et d'explorer les données ingérées dans la couche Bronze. 
**Objectif :** Vérifier la qualité des données, identifier les anomalies (valeurs nulles, doublons) avant de passer à la couche Silver.

In [3]:
import os
from src.common.spark_session_manager import get_spark_session
from src.config import NIVEAU_VIE_PAUVRETE_BRONZE_PATH
from pyspark.sql.functions import col

# 1. Initialiser la session Spark
spark = get_spark_session(app_name="Exploration_Bronze")

# 2. Configuration "Look Databricks" (Eager Evaluation)
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)

## 📊 Exploration de Niveau de vie et pauvreté 2019 Lyon au km carroyé
Lecture des données depuis le format **Parquet** (stockage optimisé).

In [4]:
nvp_path = os.path.join(NIVEAU_VIE_PAUVRETE_BRONZE_PATH, "2019_carreaux_1km_met")
df_nvp_2019 = spark.read.parquet(nvp_path)

# Aperçu des données
df_nvp_2019

26/03/08 21:57:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


idcar_1km,i_est_1km,lcog_geo,ind,men,men_pauv,men_1ind,men_5ind,men_prop,men_fmp,ind_snv,men_surf,men_coll,men_mais,log_av45,log_45_70,log_70_90,log_ap90,log_inc,log_soc,ind_0_3,ind_4_5,ind_6_10,ind_11_17,ind_18_24,ind_25_39,ind_40_54,ind_55_64,ind_65_79,ind_80p,ind_inc,source_file,bronze_processing_timestamp
CRS3035RES1000mN2...,0,5114851648,41.5,20,3,9,1,13,2,915573,2186,3.1,16.9,6,3.1,4,6.9,0,0,1.1,1.1,2,3.4,4.9,4,6,10,6,3,0,2019_carreaux_1km...,2026-03-08 15:45:...
CRS3035RES1000mN2...,0,51141,219,85.9,5.1,25,7.9,77.1,5.9,5436115.4,10191.1,3,82.9,24.1,13.8,27.9,20.1,0,0,11,2.9,15,22,12.9,39,42,28.2,35.1,10.9,0,2019_carreaux_1km...,2026-03-08 15:45:...
CRS3035RES1000mN2...,0,51141,111,37.9,1,6,5,36,2.1,2574851,4040,0,37.9,0,0,13.9,24,0,0,7,5,9,14.9,2.1,25,21,13,11,3,0,2019_carreaux_1km...,2026-03-08 15:45:...
CRS3035RES1000mN2...,1,5158951472,1,0.4,0,0.2,0,0.4,0,25511.1,52.8,0,0.4,0.1,0.1,0.1,0.1,0,0,0,0,0.1,0.2,0,0.1,0.3,0.1,0.1,0.1,0,2019_carreaux_1km...,2026-03-08 15:45:...
CRS3035RES1000mN2...,1,5159051589,3,1.3,0.2,0.4,0.1,0.9,0.1,65057.6,165.7,0.2,1.1,0.4,0.4,0.2,0.3,0,0.1,0.1,0.2,0.2,0.3,0.2,0.4,0.5,0.5,0.4,0.2,0,2019_carreaux_1km...,2026-03-08 15:45:...
CRS3035RES1000mN2...,1,51608,78,36.8,6.6,12.9,0.9,33.9,7.4,1793214.4,4158.6,0,36.8,21.2,3.5,3.9,8.2,0,0,0.9,2.7,3.8,7.3,6.5,10.9,13.8,10,16.5,5.6,0,2019_carreaux_1km...,2026-03-08 15:45:...
CRS3035RES1000mN2...,1,55304,97.5,48,4.8,20.2,1,38,4.8,2139460.2,5312.8,1,47,34.9,2.7,4.7,5.7,0,0,3.8,3.3,1.9,7.9,4.8,18.5,16.4,16.2,15.1,9.6,0,2019_carreaux_1km...,2026-03-08 15:45:...
CRS3035RES1000mN2...,0,55304,63,28.1,0,8,1,27.1,1,1516142.3,3148,1,27.1,4.1,0,16,8,0,0,0.9,2,3,4.1,6,5,13.9,6.2,15,6.9,0,2019_carreaux_1km...,2026-03-08 15:45:...
CRS3035RES1000mN2...,1,55123,18,7.9,0.5,1.8,0.5,7,0,507156.4,986.4,0,7.9,4.2,0.9,1.3,1.5,0,0,0.4,0,0.4,1.8,0.9,2.3,2.4,2.8,6,1,0,2019_carreaux_1km...,2026-03-08 15:45:...
CRS3035RES1000mN2...,0,55123,28,14.9,4,5,0,13.1,0.9,597617.3,1590,0,14.9,12.9,0,2,0,0,0,0.9,0,0.9,1,1.1,3,5,1.1,12,3,0,2019_carreaux_1km...,2026-03-08 15:45:...


## 🔬 Analyse Technique des Schémas (Data Profiling)
Il est crucial de vérifier les types détectés par Spark pour anticiper le nettoyage en couche Silver.

In [5]:
# Affichage des types de colonnes (Senior Approach)
print("--- Analyse des types de données ---")
for col_name, dtype in df_nvp_2019.dtypes:
    if "string" in dtype:
        print(f"⚠️ Colonne TEXTE (potentiel nettoyage requis) : {col_name}")
    else:
        print(f"✅ Colonne NUMÉRIQUE : {col_name}")

--- Analyse des types de données ---
⚠️ Colonne TEXTE (potentiel nettoyage requis) : idcar_1km
⚠️ Colonne TEXTE (potentiel nettoyage requis) : i_est_1km
⚠️ Colonne TEXTE (potentiel nettoyage requis) : lcog_geo
⚠️ Colonne TEXTE (potentiel nettoyage requis) : ind
⚠️ Colonne TEXTE (potentiel nettoyage requis) : men
⚠️ Colonne TEXTE (potentiel nettoyage requis) : men_pauv
⚠️ Colonne TEXTE (potentiel nettoyage requis) : men_1ind
⚠️ Colonne TEXTE (potentiel nettoyage requis) : men_5ind
⚠️ Colonne TEXTE (potentiel nettoyage requis) : men_prop
⚠️ Colonne TEXTE (potentiel nettoyage requis) : men_fmp
⚠️ Colonne TEXTE (potentiel nettoyage requis) : ind_snv
⚠️ Colonne TEXTE (potentiel nettoyage requis) : men_surf
⚠️ Colonne TEXTE (potentiel nettoyage requis) : men_coll
⚠️ Colonne TEXTE (potentiel nettoyage requis) : men_mais
⚠️ Colonne TEXTE (potentiel nettoyage requis) : log_av45
⚠️ Colonne TEXTE (potentiel nettoyage requis) : log_45_70
⚠️ Colonne TEXTE (potentiel nettoyage requis) : log_70_90
⚠️

## 🔍 Statistiques Descriptives
On se concentre sur les colonnes de participation pour vérifier la cohérence des chiffres (Inscrits, Votants, Exprimés).

In [7]:
numeric_cols = ["Ind", "Men_1ind", "Men_5ind", "Men_prop", "Men_fmp", "Ind_snv", "Men_surf",
                "Men_coll", "Men_mais", "Log_av45", "Log_45_70", "Log_70_90", "Log_ap90", "Log_inc",
                "Log_soc", "Ind_0_3", "Ind_4_5", "Ind_6_10", "Ind_11_17", "Ind_18_24", "Ind_25_39",
                "Ind_40_54", "Ind_55_64", "Ind_65_79", "Ind_80p", "Ind_inc","Men_pauv",	"Men"]
df_nvp_2019.select(numeric_cols).describe().toPandas()

,summary,Ind,Men_1ind,Men_5ind,Men_prop,Men_fmp,Ind_snv,Men_surf,Men_coll,Men_mais,...,Ind_11_17,Ind_18_24,Ind_25_39,Ind_40_54,Ind_55_64,Ind_65_79,Ind_80p,Ind_inc,Men_pauv,Men
0,count,374027,374027,374027,374027,374027,374027,374027,374027,374027,...,374027,374027,374027,374027,374027,374027,374027,374027,374027,374027
1,mean,168.35969863138223,26.544024896598483,4.883633267117125,43.97262577300558,8.234831977370565,4040783.012662467,6540.143772508428,32.25680900042003,42.43623160894784,...,14.641907669767008,12.452206124156685,30.513636716065914,33.35367473471163,21.547882104767755,24.34002518534757,9.681756397265037,0.19975723677702398,10.559826429642614,74.69304060936763
2,stddev,787.1594627732344,172.48737824634264,27.378942775191305,159.67799469291057,43.936213879134904,2.1557175470867705E7,25336.818767214914,322.9181112003802,106.23586076204414,...,65.55266107616553,59.620648333059215,182.72091285088115,155.5041628054654,89.9727689370082,99.75040903681291,44.30386038634891,3.231704280525441,67.09952133132754,373.31649077220675
3,min,1,0,0,0,0,10000020.5,100,0,0,...,0,0,0,0,0,0,0,0,0,0.2
4,max,999.5,999,99.9,999,999,99999.9,9999,999.9,999,...,998.5,998,999,999.5,999,998.1,999,99,999,999.9
